In [1]:
%pip install ultralytics ensemble-boxes -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 5.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import torch
from pathlib import Path

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

candidates = [
    "/kaggle/input/cxr-yolo-8class-dataset/dataset",
    "/kaggle/input/cxr-yolo-8class-dataset",
    "/kaggle/input/datasets/fatemadev/cxr-yolo-8class-dataset/dataset",
    "/kaggle/input/notebooks/fatemadev/notebook4fd2efe8c7/dataset",
]

DATASET_ROOT = None
for c in candidates:
    if Path(c).exists():
        train_dir = Path(c) / "images/train"
        if train_dir.exists():
            n = len(list(train_dir.glob("*.png")))
            if n > 100:
                DATASET_ROOT = c
                print(f" Dataset mila: {c}")
                print(f" Train images: {n}")
                break

if DATASET_ROOT is None:
    print(" Still not found. Manual check:")
    for root, dirs, files in os.walk("/kaggle/input/cxr-yolo-8class-dataset"):
        level = root.count(os.sep) - "/kaggle/input/cxr-yolo-8class-dataset".count(os.sep)
        if level > 3:
            continue
        print(f"{'  '*level}{Path(root).name}/")
else:
    print(f"\n DATASET_ROOT = {DATASET_ROOT}")

GPU: Tesla T4
 Dataset mila: /kaggle/input/datasets/fatemadev/cxr-yolo-8class-dataset/dataset
 Train images: 3066

 DATASET_ROOT = /kaggle/input/datasets/fatemadev/cxr-yolo-8class-dataset/dataset


In [3]:
# Structure verify
for split in ["train", "val", "test"]:
    img_dir = Path(DATASET_ROOT) / f"images/{split}"
    lbl_dir = Path(DATASET_ROOT) / f"labels/{split}"
    n_img = len(list(img_dir.glob("*.png"))) if img_dir.exists() else 0
    n_lbl = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0
    print(f"{split}: {n_img} images, {n_lbl} labels")

# Pehli image ka sample dekhein
sample = sorted((Path(DATASET_ROOT) / "images/train").glob("*.png"))[0]
print(f"\nSample: {sample.name}")
print(f"Size: {sample.stat().st_size / 1024:.1f} KB")

train: 3066 images, 3066 labels
val: 657 images, 657 labels
test: 657 images, 657 labels

Sample: 0005e8e3701dfb1dd93d53e2ff537b6e.png
Size: 2651.8 KB


In [4]:
import yaml

TRAIN_ABS = f"{DATASET_ROOT}/images/train"
VAL_ABS   = f"{DATASET_ROOT}/images/val"
TEST_ABS  = f"{DATASET_ROOT}/images/test"

train_yaml = {
    'path': '/kaggle/working',   
    'train': TRAIN_ABS,           
    'val': VAL_ABS,
    'test': TEST_ABS,
    'nc': 8,
    'names': [
        'Aortic enlargement',
        'Cardiomegaly',
        'Pleural thickening',
        'Pulmonary fibrosis',
        'Lung Opacity',
        'Other lesion',
        'Pleural effusion',
        'Nodule/Mass',
    ]
}

with open('/kaggle/working/dataset_full.yaml', 'w') as f:
    yaml.dump(train_yaml, f, default_flow_style=False, sort_keys=False)

print(" Fixed dataset_full.yaml:")
print(open('/kaggle/working/dataset_full.yaml').read())

 Fixed dataset_full.yaml:
path: /kaggle/working
train: /kaggle/input/datasets/fatemadev/cxr-yolo-8class-dataset/dataset/images/train
val: /kaggle/input/datasets/fatemadev/cxr-yolo-8class-dataset/dataset/images/val
test: /kaggle/input/datasets/fatemadev/cxr-yolo-8class-dataset/dataset/images/test
nc: 8
names:
- Aortic enlargement
- Cardiomegaly
- Pleural thickening
- Pulmonary fibrosis
- Lung Opacity
- Other lesion
- Pleural effusion
- Nodule/Mass



In [5]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data='/kaggle/working/dataset_full.yaml',
    epochs=100,
    imgsz=1024,
    batch=8,
    device=0,
    workers=2,
    project='/kaggle/working/runs',
    name='full_yolov8s_1024',
    exist_ok=True,
    plots=True,
    verbose=True,
    
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    cos_lr=True,
    
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10.0, translate=0.1, scale=0.5, shear=2.0,
    perspective=0.0,
    flipud=0.0, fliplr=0.5,
    mosaic=1.0, mixup=0.1,
    
    cls=0.7,
    patience=20,
    save_period=10,
    seed=42,
    deterministic=True,
    
    amp=True,
    cache=True,
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.164 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.7, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/dataset_full.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


      2/100      6.21G      2.022      3.746      2.035         19       1024: 100% ━━━━━━━━━━━━ 384/384 4.6it/s 1:24
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 42/42 5.2it/s 8.1s
                   all        657       2959      0.259      0.283      0.232      0.115

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/100      6.21G       1.98      3.553      1.979          7       1024: 100% ━━━━━━━━━━━━ 384/384 4.6it/s 1:24
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 42/42 4.6it/s 9.2s
                   all        657       2959      0.296      0.291      0.254      0.122

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      4/100      6.21G      1.937      3.417      1.918         18       1024: 100% ━━━━━━━━━━━━ 384/384 4.6it/s 1:24
                 Class     Images  Instances      Box(P 